In [ ]:
SID4 = 9486
SEED = 9486
SLICE = 486
HP_ID = 0
CLS_A = 6
CLS_B = 0

for name, value in {'SID4': SID4, 'SEED': SEED, 'SLICE': SLICE, 'HP_ID': HP_ID, 'CLS_A': CLS_A, 'CLS_B': CLS_B}.items():
    print(f'{name} = {value}')
print('SEED=9486 controls stochastic operations.')
print('SLICE=486 and HP_ID=0 are reported only; HW2 has no HP_ID mapping.')
print('CLS_A=6 and CLS_B=0 are reported only because standing requirements require them.')


# DATA 266 — Homework 2: NLP and Deep Learning Optimization

Parts 1 and 2 follow a straightforward, sequential Google Colab workflow. Part 3 is reserved for later.

## 1. Personal Parameters and Reproducibility

Run this setup before the experiment cells.

In [ ]:
import os
import random
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)

import numpy as np
np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        torch.cuda.manual_seed_all(SEED)
except ImportError:
    print('PyTorch is unavailable.')

try:
    import tensorflow as tf
    tf.keras.utils.set_random_seed(SEED)
except ImportError:
    print('TensorFlow is unavailable.')

print(f'PYTHONHASHSEED={os.environ["PYTHONHASHSEED"]}; stochastic seed={SEED}')


---
# Part 1 — Embedding-Based Transfer Learning (Word2Vec + IMDB)

Start with Google News vectors, inspect neighbors, then fine-tune a new Word2Vec model on local IMDB reviews. Neighbor similarity compares words; original-versus-fine-tuned cosine similarity measures a target word's own movement.

### 1.1 Load Packages and 1.2 Load the Pretrained Model

Run the commented installation line in Colab if needed. The Word2Vec download occurs only when the model-loading cell is intentionally run.

In [ ]:
# !pip install -q gensim scikit-learn matplotlib pandas numpy
import gc
import json
import re
import matplotlib.pyplot as plt
import pandas as pd
from gensim.models import Word2Vec
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity
import gensim.downloader as api

pretrained_kv = api.load('word2vec-google-news-300')
print('Vocabulary size:', len(pretrained_kv.key_to_index))
print('Vector size:', pretrained_kv.vector_size)


### 1.3 Baseline Nearest Neighbors

Extract top-three cosine-similarity neighbors for the required words.

In [ ]:
TARGET_WORDS = ['cast', 'score', 'plot', 'screen', 'review']

def top_neighbors(kv_model, words, stage, topn=3):
    rows = []
    for word in words:
        for rank, (neighbor, similarity) in enumerate(kv_model.most_similar(word, topn=topn), start=1):
            rows.append({'word': word, 'rank': rank, f'neighbor_{stage}': neighbor, f'similarity_{stage}': float(similarity)})
    return pd.DataFrame(rows)

baseline_neighbors = top_neighbors(pretrained_kv, TARGET_WORDS, 'before')
baseline_neighbors


### 1.4 Load and Tokenize the Local IMDB CSV

The dataset is loaded directly from `data/IMDB Dataset.csv`. Text is lowercased, HTML break tags are removed, punctuation is separated, and whitespace tokens are returned.

In [ ]:
imdb_df = pd.read_csv('data/IMDB Dataset.csv')
assert {'review', 'sentiment'}.issubset(imdb_df.columns), 'Expected review and sentiment columns.'

def simple_tokenize(text):
    text = str(text).lower()
    text = re.sub(r'<br\s*/?\s*>', ' ', text)
    text = re.sub(r"[^a-z0-9']+", ' ', text)
    return text.split()

N_REVIEWS = len(imdb_df)
imdb_sentences = [simple_tokenize(text) for text in imdb_df['review'].iloc[:N_REVIEWS]]
print('Dataset path: data/IMDB Dataset.csv')
print('Dataset shape:', imdb_df.shape)
print('Reviews used:', len(imdb_sentences))
print('Example tokenized review:', imdb_sentences[0][:20])


### 1.5 Fine-Tune Word2Vec

A new IMDB vocabulary is built. Shared words receive copies of original pretrained vectors, so the original model remains unchanged. `workers=1` and `SEED=9486` support reproducibility, though Word2Vec can still vary by environment.

In [ ]:
WINDOW = 5
MIN_COUNT = 2
EPOCHS = 5
finetuned_model = Word2Vec(vector_size=pretrained_kv.vector_size, window=WINDOW, min_count=MIN_COUNT, workers=1, seed=SEED, sorted_vocab=1, epochs=EPOCHS)
finetuned_model.build_vocab(imdb_sentences)
shared_words = [word for word in finetuned_model.wv.index_to_key if word in pretrained_kv.key_to_index]
for word in shared_words:
    finetuned_model.wv[word] = pretrained_kv[word].copy()
print(f'Initialized {len(shared_words)} shared IMDB words from pretrained vectors.')
finetuned_model.train(imdb_sentences, total_examples=finetuned_model.corpus_count, epochs=finetuned_model.epochs)


### 1.6–1.9 Post-Fine-Tuning Neighbors, Comparison, Movement, and t-SNE

Lower original-versus-fine-tuned cosine similarity means greater movement in embedding space.

In [ ]:
finetuned_neighbors = top_neighbors(finetuned_model.wv, TARGET_WORDS, 'after')
neighbor_comparison = baseline_neighbors.merge(finetuned_neighbors, on=['word', 'rank'])
display(finetuned_neighbors)
display(neighbor_comparison)

shift_rows = []
for word in TARGET_WORDS:
    similarity = float(cosine_similarity(pretrained_kv[word].reshape(1, -1), finetuned_model.wv[word].reshape(1, -1))[0][0])
    shift_rows.append({'word': word, 'cosine_similarity_original_vs_finetuned': similarity, 'shift': 1 - similarity})
shift_df = pd.DataFrame(shift_rows).sort_values('shift', ascending=False).reset_index(drop=True)
display(shift_df)

most_shifted = shift_df.iloc[0]
least_shifted = shift_df.iloc[-1]
print(f"Most shifted: {most_shifted['word']} | similarity={most_shifted['cosine_similarity_original_vs_finetuned']:.4f} | shift={most_shifted['shift']:.4f}")
print(f"Least shifted: {least_shifted['word']} | similarity={least_shifted['cosine_similarity_original_vs_finetuned']:.4f} | shift={least_shifted['shift']:.4f}")

VIZ_WORD = 'score'
before_words = [VIZ_WORD] + [word for word, _ in pretrained_kv.most_similar(VIZ_WORD, topn=5)]
after_words = [VIZ_WORD] + [word for word, _ in finetuned_model.wv.most_similar(VIZ_WORD, topn=5)]
vectors = np.vstack([[pretrained_kv[word] for word in before_words], [finetuned_model.wv[word] for word in after_words]])
coordinates = TSNE(n_components=2, random_state=SEED, perplexity=5, init='pca').fit_transform(vectors)
plt.figure(figsize=(10, 8))
n_before = len(before_words)
plt.scatter(coordinates[:n_before, 0], coordinates[:n_before, 1], color='tab:blue', label='Before fine-tuning')
plt.scatter(coordinates[n_before:, 0], coordinates[n_before:, 1], color='tab:orange', label='After fine-tuning')
for index, word in enumerate(before_words):
    plt.annotate(f'{word} (before)', coordinates[index], fontsize=8)
for index, word in enumerate(after_words, start=n_before):
    plt.annotate(f'{word} (after)', coordinates[index], fontsize=8)
plt.title("t-SNE: neighbors of 'score' before vs. after IMDB fine-tuning")
plt.legend()
plt.tight_layout()
os.makedirs('figures/hw2', exist_ok=True)
plt.savefig('figures/hw2/part1_embedding_shift_tsne.png', dpi=200, bbox_inches='tight')
plt.show()


### 1.10 Save Part 1 Artifacts

Run only after actual Part 1 results are available.

In [ ]:
os.makedirs('artifacts/part1', exist_ok=True)
baseline_neighbors.to_csv('artifacts/part1/baseline_neighbors.csv', index=False)
finetuned_neighbors.to_csv('artifacts/part1/finetuned_neighbors.csv', index=False)
neighbor_comparison.to_csv('artifacts/part1/neighbor_comparison.csv', index=False)
shift_df.to_csv('artifacts/part1/vector_shift_metrics.csv', index=False)
with open('artifacts/part1/part1_configuration.json', 'w') as file:
    json.dump({'seed': SEED, 'dataset_path': 'data/IMDB Dataset.csv', 'reviews_used': N_REVIEWS, 'window': WINDOW, 'min_count': MIN_COUNT, 'epochs': EPOCHS, 'shared_words_initialized': len(shared_words)}, file, indent=2)
with open('artifacts/part1/part1_run_summary.txt', 'w') as file:
    file.write(f"Most shifted: {most_shifted['word']} | similarity={most_shifted['cosine_similarity_original_vs_finetuned']:.6f} | shift={most_shifted['shift']:.6f}\n")
    file.write(f"Least shifted: {least_shifted['word']} | similarity={least_shifted['cosine_similarity_original_vs_finetuned']:.6f} | shift={least_shifted['shift']:.6f}\n")


---
# Part 2 — LangChain RAG over Wikipedia Movie Pages

The explicit pipeline uses `WikipediaLoader`, chunking, `sentence-transformers/all-MiniLM-L6-v2`, FAISS, a retriever, a context formatter, `PromptTemplate`, and `google/flan-t5-base`.

### 2.1 Install/Import Packages and Load the Local LLM

The selected LLM is exactly `google/flan-t5-base`; the selected embedding model is exactly `sentence-transformers/all-MiniLM-L6-v2`.

In [ ]:
# !pip install -q langchain langchain-community langchain-huggingface langchain-text-splitters faiss-cpu wikipedia sentence-transformers transformers accelerate
import torch
from langchain_community.document_loaders import WikipediaLoader
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import pipeline, set_seed

set_seed(SEED)
generator = pipeline('text2text-generation', model='google/flan-t5-base', device=0 if torch.cuda.is_available() else -1, max_new_tokens=128, do_sample=False)
llm = HuggingFacePipeline(pipeline=generator)
print('LLM: google/flan-t5-base')


### 2.2 Load Ten Wikipedia Movie Pages

The requested titles, URLs, metadata, and exact loaded text are retained.

In [ ]:
MOVIES = ['Inception', 'The Godfather', 'Titanic (1997 film)', 'The Dark Knight', 'Pulp Fiction', 'Forrest Gump', 'The Matrix', 'Interstellar', 'Parasite (2019 film)', 'Gladiator (2000 film)']
documents = []
for title in MOVIES:
    document = WikipediaLoader(query=title, load_max_docs=1, doc_content_chars_max=20000).load()[0]
    document.metadata['requested_movie_title'] = title
    documents.append(document)
print(f'Loaded {len(documents)} documents')
print(documents[0].metadata)

os.makedirs('artifacts/part2', exist_ok=True)
with open('artifacts/part2/cached_documents.json', 'w') as file:
    json.dump([{'page_content': document.page_content, 'metadata': document.metadata} for document in documents], file, indent=2, ensure_ascii=False, default=str)


### 2.3 Split Chunks (500 / 50), Embed, and Build FAISS

In [ ]:
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
def split_documents(docs, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP):
    return RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap).split_documents(docs)

chunks = split_documents(documents)
print(f'Total chunks: {len(chunks)}')
print(chunks[0].page_content[:300])
pd.DataFrame([{'chunk_id': index + 1, 'title': chunk.metadata.get('title'), 'source': chunk.metadata.get('source'), 'requested_movie_title': chunk.metadata.get('requested_movie_title'), 'page_content': chunk.page_content} for index, chunk in enumerate(chunks)]).to_csv('artifacts/part2/chunk_manifest.csv', index=False)

EMBEDDING_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL, model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'}, encode_kwargs={'normalize_embeddings': True})
vectorstore = FAISS.from_documents(chunks, embeddings)
print(f'Embedding model: {EMBEDDING_MODEL}')


### 2.4 Explicit PromptTemplate, Retriever, Context Formatter, and LLM

In [ ]:
prompt_template = PromptTemplate(input_variables=['context', 'question'], template='Answer using only the context below. If the answer is not in the context, say you do not know.\n\nContext:\n{context}\n\nQuestion: {question}\nAnswer:')

def format_context(retrieved_docs):
    return '\n\n'.join(f"[Rank {rank} | {document.metadata.get('title')} | {document.metadata.get('source')}]\n{document.page_content}" for rank, document in enumerate(retrieved_docs, start=1))

def answer_question(question, store, k=3):
    retriever = store.as_retriever(search_kwargs={'k': k})
    retrieved_docs = retriever.invoke(question)
    prompt = prompt_template.format(context=format_context(retrieved_docs), question=question)
    response = llm.invoke(prompt)
    return (response.content if hasattr(response, 'content') else str(response)), retrieved_docs


### 2.5 Retrieve Top-3 Chunks and Answer Five Questions

Each result displays complete retrieved passages, rank, title, URL, and generated answer.

In [ ]:
QUESTIONS = ['Who directed Inception, and what is its central plot concept?', 'Who played Don Vito Corleone in The Godfather?', 'What ship is central to the story of Titanic?', 'Who is the main antagonist in The Dark Knight?', 'What major Oscar award did Parasite win?']
results = []
for question in QUESTIONS:
    answer, retrieved_docs = answer_question(question, vectorstore, k=3)
    results.append({'question': question, 'answer': answer, 'retrieved_docs': retrieved_docs})
    print(f'Question: {question}\nAnswer: {answer}\nRetrieved chunks:')
    for rank, document in enumerate(retrieved_docs, start=1):
        print(f"\n[Rank {rank}] {document.metadata.get('title')} | {document.metadata.get('source')}\n{document.page_content}")
    print('\n' + '-' * 100)


### 2.6 Re-run Exactly Two Questions with Alternate Chunking (800 / 100)

In [ ]:
ALT_CHUNK_SIZE = 800
ALT_CHUNK_OVERLAP = 100
QUESTIONS_TO_RECOMPARE = QUESTIONS[:2]
alt_chunks = split_documents(documents, ALT_CHUNK_SIZE, ALT_CHUNK_OVERLAP)
alt_vectorstore = FAISS.from_documents(alt_chunks, embeddings)
comparison_rows = []
for question in QUESTIONS_TO_RECOMPARE:
    original = next(result for result in results if result['question'] == question)
    alt_answer, alt_docs = answer_question(question, alt_vectorstore, k=3)
    comparison_rows.append({'question': question, 'original_chunk_configuration': f'{CHUNK_SIZE}/{CHUNK_OVERLAP}', 'alternate_chunk_configuration': f'{ALT_CHUNK_SIZE}/{ALT_CHUNK_OVERLAP}', 'original_retrieved_sources': ' | '.join(doc.metadata.get('source', '') for doc in original['retrieved_docs']), 'alternate_retrieved_sources': ' | '.join(doc.metadata.get('source', '') for doc in alt_docs), 'original_answer': original['answer'], 'alternate_answer': alt_answer, 'answer_changed': original['answer'].strip() != alt_answer.strip(), 'interpretation': 'Inspect retrieved passages and answers after execution.'})
chunking_comparison = pd.DataFrame(comparison_rows)
chunking_comparison


### 2.7 Retrieval Success and 2.8 Failure Analysis

Use the complete displayed passages for manual verification. The keyword check is only an aid: fill the source passage, movie, rank, and explanation from actual retrieved text before reporting results. Do not invent the two failure analyses; replace the prompts only after actual evidence is available.

In [ ]:
EXPECTED_KEYWORDS = {QUESTIONS[0]: 'christopher nolan', QUESTIONS[1]: 'marlon brando', QUESTIONS[2]: 'rms titanic', QUESTIONS[3]: 'joker', QUESTIONS[4]: 'best picture'}
evaluation_rows = []
for result in results:
    keyword = EXPECTED_KEYWORDS[result['question']]
    rank = next((rank for rank, document in enumerate(result['retrieved_docs'], start=1) if keyword in document.page_content.lower()), None)
    evaluation_rows.append({'question': result['question'], 'manually_identified_correct_source_passage': 'Complete after manual inspection.', 'relevant_movie': 'Complete after manual inspection.', 'top3_contains_required_information': rank is not None, 'rank_of_first_relevant_chunk': rank, 'explanation': 'Keyword-assisted; manually verify complete passages.'})
retrieval_evaluation = pd.DataFrame(evaluation_rows)
display(retrieval_evaluation)
success_count = retrieval_evaluation['top3_contains_required_information'].sum()
retrieval_success_rate = success_count / len(QUESTIONS)
print(f'Retrieval Success Rate: {success_count}/{len(QUESTIONS)} = {retrieval_success_rate:.2%}')


1. **Failure analysis 1:** [Complete after inspecting actual retrieved passages and generated answer.]
2. **Failure analysis 2:** [Complete after inspecting actual retrieved passages and generated answer.]

### 2.9 Save Part 2 Artifacts

In [ ]:
retrieval_records = [{'question': result['question'], 'generated_answer': result['answer'], 'retrieved_chunks': [{'rank': rank, 'title': document.metadata.get('title'), 'source': document.metadata.get('source'), 'passage': document.page_content} for rank, document in enumerate(result['retrieved_docs'], start=1)]} for result in results]
with open('artifacts/part2/retrieval_results.json', 'w') as file:
    json.dump(retrieval_records, file, indent=2, ensure_ascii=False)
retrieval_evaluation.to_csv('artifacts/part2/retrieval_evaluation.csv', index=False)
chunking_comparison.to_csv('artifacts/part2/chunking_comparison.csv', index=False)
with open('artifacts/part2/rag_configuration.json', 'w') as file:
    json.dump({'seed': SEED, 'movies': MOVIES, 'primary_chunking': [CHUNK_SIZE, CHUNK_OVERLAP], 'alternate_chunking': [ALT_CHUNK_SIZE, ALT_CHUNK_OVERLAP], 'top_k': 3, 'embedding_model': EMBEDDING_MODEL, 'vectorstore': 'FAISS', 'llm': 'google/flan-t5-base', 'questions': QUESTIONS, 'alternate_questions': QUESTIONS_TO_RECOMPARE}, file, indent=2)


---
# Part 3 — Optimization Techniques

These direct PyTorch comparisons use the same fixed synthetic dataset, seed, optimizer, learning rate, batch size, and number of steps whenever the technique permits a matched baseline. Results are created only when these cells are run.

### 3.1 Shared Setup

The fixed dataset and settings below are reused across the training comparisons.

In [ ]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
INPUT_DIM = 128
HIDDEN_DIM = 256
OUTPUT_DIM = 10
BATCH_SIZE = 64
NUM_STEPS = 40
LEARNING_RATE = 1e-3

X = torch.randn(BATCH_SIZE * NUM_STEPS, INPUT_DIM)
y = torch.randint(0, OUTPUT_DIM, (BATCH_SIZE * NUM_STEPS,))

print('PyTorch version:', torch.__version__)
print('Selected device:', DEVICE)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
print({'input_dim': INPUT_DIM, 'hidden_dim': HIDDEN_DIM, 'output_dim': OUTPUT_DIM, 'batch_size': BATCH_SIZE, 'steps': NUM_STEPS, 'learning_rate': LEARNING_RATE, 'optimizer': 'Adam', 'seed': SEED})

def get_batches():
    for step in range(NUM_STEPS):
        start = step * BATCH_SIZE
        end = start + BATCH_SIZE
        yield X[start:end].to(DEVICE), y[start:end].to(DEVICE)

def start_measurement():
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

def finish_measurement():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
        return torch.cuda.max_memory_allocated() / (1024 ** 2)
    return float('nan')


### 3.2 Technique 1 — Tensor Creation: CPU versus GPU

CUDA synchronization brackets GPU timing. If CUDA is unavailable, only the measured CPU result is recorded and the GPU comparison is explicitly unavailable.

In [ ]:
TENSOR_SHAPE = (4096, 512)
TENSOR_ITERATIONS = 25
tensor_results = []

start = time.perf_counter()
for _ in range(TENSOR_ITERATIONS):
    cpu_tensor = torch.randn(TENSOR_SHAPE, device='cpu')
cpu_seconds = time.perf_counter() - start
tensor_results.append({'technique': 'Tensor creation', 'configuration': 'CPU tensor creation', 'device': 'cpu', 'batch_size': None, 'effective_batch_size': None, 'training_steps': TENSOR_ITERATIONS, 'time_seconds': cpu_seconds, 'peak_memory_mb': float('nan'), 'final_loss': float('nan'), 'notes': f'tensor shape={TENSOR_SHAPE}'})
print(f'CPU tensor creation: {cpu_seconds:.4f}s | shape={tuple(cpu_tensor.shape)}')

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(TENSOR_ITERATIONS):
        gpu_tensor = torch.randn(TENSOR_SHAPE, device='cuda')
    torch.cuda.synchronize()
    gpu_seconds = time.perf_counter() - start
    gpu_memory = torch.cuda.max_memory_allocated() / (1024 ** 2)
    tensor_results.append({'technique': 'Tensor creation', 'configuration': 'GPU tensor creation', 'device': 'cuda', 'batch_size': None, 'effective_batch_size': None, 'training_steps': TENSOR_ITERATIONS, 'time_seconds': gpu_seconds, 'peak_memory_mb': gpu_memory, 'final_loss': float('nan'), 'notes': f'tensor shape={TENSOR_SHAPE}'})
    print(f'GPU tensor creation: {gpu_seconds:.4f}s | shape={tuple(gpu_tensor.shape)} | peak memory={gpu_memory:.2f} MB')
else:
    print('CUDA is unavailable: GPU tensor-creation comparison is unavailable.')


### 3.3 Shared Training Models

`SimpleMLP` is used for weight initialization, gradient accumulation, and mixed precision. `DeepMLP` is used for the checkpointing comparison.

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self, init_fn=None):
        super().__init__()
        self.fc1 = nn.Linear(INPUT_DIM, HIDDEN_DIM)
        self.fc2 = nn.Linear(HIDDEN_DIM, HIDDEN_DIM)
        self.fc3 = nn.Linear(HIDDEN_DIM, OUTPUT_DIM)
        if init_fn is not None:
            for layer in [self.fc1, self.fc2, self.fc3]:
                init_fn(layer.weight)

    def forward(self, inputs):
        inputs = F.relu(self.fc1(inputs))
        inputs = F.relu(self.fc2(inputs))
        return self.fc3(inputs)

class DeepMLP(nn.Module):
    def __init__(self, use_checkpointing=False):
        super().__init__()
        self.use_checkpointing = use_checkpointing
        self.input_layer = nn.Linear(INPUT_DIM, HIDDEN_DIM)
        self.hidden_layers = nn.ModuleList([nn.Linear(HIDDEN_DIM, HIDDEN_DIM) for _ in range(8)])
        self.output_layer = nn.Linear(HIDDEN_DIM, OUTPUT_DIM)

    def hidden_block(self, inputs):
        for layer in self.hidden_layers:
            inputs = F.relu(layer(inputs))
        return inputs

    def forward(self, inputs):
        inputs = F.relu(self.input_layer(inputs))
        if self.use_checkpointing:
            inputs = checkpoint(self.hidden_block, inputs, use_reentrant=False)
        else:
            inputs = self.hidden_block(inputs)
        return self.output_layer(inputs)


### 3.4 Technique 2 — Weight Initialization

Compare PyTorch default initialization with Xavier initialization. Both runs reset to `SEED=9486`, use the same fixed data, and train for the same number of steps.

In [ ]:
def run_weight_initialization(init_fn, label):
    torch.manual_seed(SEED)
    model = SimpleMLP(init_fn=init_fn).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    start_measurement()
    start = time.perf_counter()
    losses = []
    for xb, yb in get_batches():
        optimizer.zero_grad()
        loss = F.cross_entropy(model(xb), yb)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    elapsed = time.perf_counter() - start
    memory = finish_measurement()
    return {'technique': 'Weight initialization', 'configuration': label, 'device': str(DEVICE), 'batch_size': BATCH_SIZE, 'effective_batch_size': BATCH_SIZE, 'training_steps': NUM_STEPS, 'time_seconds': elapsed, 'peak_memory_mb': memory, 'final_loss': losses[-1], 'notes': 'Same seed and fixed synthetic dataset'}, losses

default_init_result, default_init_losses = run_weight_initialization(None, 'PyTorch default initialization')
xavier_init_result, xavier_init_losses = run_weight_initialization(nn.init.xavier_uniform_, 'Xavier uniform initialization')
initialization_results = [default_init_result, xavier_init_result]
pd.DataFrame(initialization_results)


### 3.5 Technique 3 — Activation Checkpointing

Checkpointing stores fewer activations and recomputes them during backpropagation. The expected tradeoff is lower GPU memory use and longer execution time.

In [ ]:
def run_checkpointing(use_checkpointing, label):
    torch.manual_seed(SEED)
    model = DeepMLP(use_checkpointing=use_checkpointing).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    start_measurement()
    start = time.perf_counter()
    losses = []
    for xb, yb in get_batches():
        optimizer.zero_grad()
        loss = F.cross_entropy(model(xb), yb)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    elapsed = time.perf_counter() - start
    memory = finish_measurement()
    return {'technique': 'Activation checkpointing', 'configuration': label, 'device': str(DEVICE), 'batch_size': BATCH_SIZE, 'effective_batch_size': BATCH_SIZE, 'training_steps': NUM_STEPS, 'time_seconds': elapsed, 'peak_memory_mb': memory, 'final_loss': losses[-1], 'notes': 'DeepMLP; checkpointing trades memory for recomputation'}, losses

no_checkpoint_result, no_checkpoint_losses = run_checkpointing(False, 'No checkpointing')
checkpoint_result, checkpoint_losses = run_checkpointing(True, 'Activation checkpointing')
checkpoint_results = [no_checkpoint_result, checkpoint_result]
pd.DataFrame(checkpoint_results)


### 3.6 Technique 4 — Gradient Accumulation

Four micro-batches are accumulated before one optimizer step. The effective batch size and total examples are the same as standard training; smaller micro-batches are expected to reduce activation memory with some loop overhead.

In [ ]:
ACCUMULATION_STEPS = 4
MICRO_BATCH_SIZE = BATCH_SIZE // ACCUMULATION_STEPS

def run_standard_training():
    torch.manual_seed(SEED)
    model = SimpleMLP().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    start_measurement()
    start = time.perf_counter()
    losses = []
    for xb, yb in get_batches():
        optimizer.zero_grad()
        loss = F.cross_entropy(model(xb), yb)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    elapsed = time.perf_counter() - start
    memory = finish_measurement()
    return {'technique': 'Gradient accumulation', 'configuration': 'Standard training', 'device': str(DEVICE), 'batch_size': BATCH_SIZE, 'effective_batch_size': BATCH_SIZE, 'training_steps': NUM_STEPS, 'time_seconds': elapsed, 'peak_memory_mb': memory, 'final_loss': losses[-1], 'notes': 'One optimizer step per full batch'}, losses

def run_gradient_accumulation():
    torch.manual_seed(SEED)
    model = SimpleMLP().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    start_measurement()
    start = time.perf_counter()
    losses = []
    for xb, yb in get_batches():
        optimizer.zero_grad()
        for micro_step in range(ACCUMULATION_STEPS):
            start_index = micro_step * MICRO_BATCH_SIZE
            end_index = start_index + MICRO_BATCH_SIZE
            loss = F.cross_entropy(model(xb[start_index:end_index]), yb[start_index:end_index]) / ACCUMULATION_STEPS
            loss.backward()
            losses.append(loss.item() * ACCUMULATION_STEPS)
        optimizer.step()
    elapsed = time.perf_counter() - start
    memory = finish_measurement()
    return {'technique': 'Gradient accumulation', 'configuration': f'{ACCUMULATION_STEPS} x {MICRO_BATCH_SIZE} micro-batches', 'device': str(DEVICE), 'batch_size': MICRO_BATCH_SIZE, 'effective_batch_size': BATCH_SIZE, 'training_steps': NUM_STEPS, 'time_seconds': elapsed, 'peak_memory_mb': memory, 'final_loss': losses[-1], 'notes': 'Same effective batch size and examples as standard training'}, losses

standard_result, standard_losses = run_standard_training()
accumulation_result, accumulation_losses = run_gradient_accumulation()
accumulation_results = [standard_result, accumulation_result]
pd.DataFrame(accumulation_results)


### 3.7 Technique 5 — Mixed Precision Training

On CUDA, compare full precision with `torch.autocast` and `torch.amp.GradScaler`. Mixed precision can lower memory use and improve throughput, while final loss should be checked against full precision. No mixed-precision measurement is created without CUDA.

In [ ]:
mixed_precision_results = []
if torch.cuda.is_available():
    def run_precision(use_mixed_precision, label):
        torch.manual_seed(SEED)
        model = SimpleMLP().to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
        scaler = torch.amp.GradScaler('cuda', enabled=use_mixed_precision)
        start_measurement()
        start = time.perf_counter()
        losses = []
        for xb, yb in get_batches():
            optimizer.zero_grad()
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_mixed_precision):
                loss = F.cross_entropy(model(xb), yb)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            losses.append(loss.item())
        elapsed = time.perf_counter() - start
        memory = finish_measurement()
        return {'technique': 'Mixed precision', 'configuration': label, 'device': 'cuda', 'batch_size': BATCH_SIZE, 'effective_batch_size': BATCH_SIZE, 'training_steps': NUM_STEPS, 'time_seconds': elapsed, 'peak_memory_mb': memory, 'final_loss': losses[-1], 'notes': 'CUDA autocast with GradScaler' if use_mixed_precision else 'CUDA full precision baseline'}, losses

    full_precision_result, full_precision_losses = run_precision(False, 'Full precision')
    mixed_precision_result, mixed_precision_losses = run_precision(True, 'CUDA mixed precision')
    mixed_precision_results = [full_precision_result, mixed_precision_result]
    display(pd.DataFrame(mixed_precision_results))
else:
    print('CUDA is unavailable: mixed precision comparison is unavailable.')


### 3.8 Final Results, Plot, and Saved Artifacts

Interpret timing, memory, and loss only after the experiment cells have run. CPU memory is recorded as unavailable (`NaN`); CUDA rows use PyTorch peak allocated memory.

In [ ]:
optimization_results = pd.DataFrame(tensor_results + initialization_results + checkpoint_results + accumulation_results + mixed_precision_results)
optimization_results

loss_results = optimization_results.dropna(subset=['final_loss'])
if not loss_results.empty:
    plt.figure(figsize=(10, 5))
    plt.bar(loss_results['configuration'], loss_results['final_loss'])
    plt.xticks(rotation=30, ha='right')
    plt.ylabel('Final loss')
    plt.title('Homework 2 Part 3 Final Loss Comparison')
    plt.tight_layout()
    os.makedirs('figures/hw2', exist_ok=True)
    plt.savefig('figures/hw2/part3_final_loss_comparison.png', dpi=200, bbox_inches='tight')
    plt.show()

os.makedirs('artifacts/part3', exist_ok=True)
optimization_results.to_csv('artifacts/part3/optimization_results.csv', index=False)
optimization_configuration = {'seed': SEED, 'pytorch_version': torch.__version__, 'device': str(DEVICE), 'cuda_available': torch.cuda.is_available(), 'input_dim': INPUT_DIM, 'hidden_dim': HIDDEN_DIM, 'output_dim': OUTPUT_DIM, 'batch_size': BATCH_SIZE, 'effective_batch_size': BATCH_SIZE, 'training_steps': NUM_STEPS, 'learning_rate': LEARNING_RATE, 'optimizer': 'Adam', 'tensor_shape': TENSOR_SHAPE, 'tensor_iterations': TENSOR_ITERATIONS, 'accumulation_steps': ACCUMULATION_STEPS, 'micro_batch_size': MICRO_BATCH_SIZE}
with open('artifacts/part3/optimization_configuration.json', 'w') as file:
    json.dump(optimization_configuration, file, indent=2)
print('Saved Part 3 results and configuration.')


## Reproducibility and Limitations

Use `SEED=9486`. Exact results can vary with library versions, hardware, Word2Vec training, and live Wikipedia content. Record observed results only after execution.